## Deep Learning Basics with PyTorch
## Part III - Supervised Deep Learning in Practice
### Chapter 9 - Working with Data in PyTorch

This practice chapter shows how data moves through a small PyTorch workflow: from a dataset, into a `DataLoader`, through a transform, into mini-batches, and finally through a simple training loop and a few practical visualizations.


In [ ]:
# Ensure required packages are available (already installed in this env)
# !pip -q install torch numpy matplotlib
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import TensorDataset, DataLoader, Dataset

%config InlineBackend.figure_format = 'retina'


## Dataset and DataLoader (moons)

A dataset stores examples and labels. A `DataLoader` turns that dataset into shuffled mini-batches so training can process a few samples at a time instead of the whole dataset at once.


In [ ]:
torch.manual_seed(0)


def make_two_moons(n_samples=600, noise=0.25, seed=0):
    generator = torch.Generator().manual_seed(seed)
    n_top = n_samples // 2
    n_bottom = n_samples - n_top

    theta_top = torch.linspace(0, torch.pi, n_top)
    theta_bottom = torch.linspace(0, torch.pi, n_bottom)

    top = torch.stack([torch.cos(theta_top), torch.sin(theta_top)], dim=1)
    bottom = torch.stack(
        [1 - torch.cos(theta_bottom), -torch.sin(theta_bottom) - 0.5], dim=1
    )

    X = torch.cat([top, bottom], dim=0)
    y = torch.cat(
        [
            torch.zeros(n_top, dtype=torch.long),
            torch.ones(n_bottom, dtype=torch.long),
        ]
    )

    X = X + noise * torch.randn(X.shape, generator=generator)
    return X, y


def split_train_test(X, y, test_size=0.25, seed=42):
    generator = torch.Generator().manual_seed(seed)
    train_idx_parts = []
    test_idx_parts = []

    for label in [0, 1]:
        idx = torch.where(y == label)[0]
        idx = idx[torch.randperm(len(idx), generator=generator)]
        split = int(len(idx) * (1 - test_size))
        train_idx_parts.append(idx[:split])
        test_idx_parts.append(idx[split:])

    train_idx = torch.cat(train_idx_parts)
    test_idx = torch.cat(test_idx_parts)
    train_idx = train_idx[torch.randperm(len(train_idx), generator=generator)]
    test_idx = test_idx[torch.randperm(len(test_idx), generator=generator)]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


X, y = make_two_moons(n_samples=600, noise=0.25, seed=0)
X_tr, X_te, y_tr, y_te = split_train_test(X, y, test_size=0.25, seed=42)

train_ds = TensorDataset(X_tr, y_tr)
test_ds = TensorDataset(X_te, y_te)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256)

xb0, yb0 = next(iter(train_loader))
print('Train dataset size:', len(train_ds))
print('Test dataset size: ', len(test_ds))
print('One mini-batch shapes:', xb0.shape, yb0.shape)


## Small training example

The dataset feeds examples into the loader, the loader yields mini-batches, and the training loop repeats that process across epochs.
Inside each mini-batch step, the optimizer updates the model with the usual sequence: forward pass, loss, `zero_grad()`, `backward()`, and `step()`.


In [ ]:
class TinyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, 2),
        )

    def forward(self, x):
        return self.net(x)


def loader_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for Xb, yb in loader:
            preds = model(Xb).argmax(dim=1)
            correct += int((preds == yb).sum())
            total += yb.size(0)
    return correct / total


model = TinyMLP()
opt = torch.optim.Adam(model.parameters(), lr=5e-3)
loss_fn = nn.CrossEntropyLoss()
epoch_loss_history = []

for epoch in range(10):
    model.train()
    running_loss = 0.0

    for Xb, yb in train_loader:
        logits = model(Xb)
        loss = loss_fn(logits, yb)

        opt.zero_grad()
        loss.backward()
        opt.step()

        running_loss += loss.item() * Xb.size(0)

    epoch_loss_history.append(running_loss / len(train_loader.dataset))

accuracy = loader_accuracy(model, test_loader)
print(f'Final test accuracy: {accuracy:.3f}')


## Transform: standardization

A transform changes each example before it is used. Here we standardize each feature with the training-set mean and standard deviation so the code stays readable and reusable.


In [ ]:
class Standardize:
    # Feature-wise standardization fitted on the training set.
    def __init__(self, mean, std, eps=1e-8):
        self.mean = mean
        self.std = std.clamp_min(eps)

    def __call__(self, x):
        return (x - self.mean) / self.std


mu = X_tr.mean(dim=0)
sigma = X_tr.std(dim=0)
standardize = Standardize(mu, sigma)

X_tr_s = standardize(X_tr)
X_te_s = standardize(X_te)

train_loader_std = DataLoader(
    TensorDataset(X_tr_s, y_tr), batch_size=64, shuffle=True
)
test_loader_std = DataLoader(
    TensorDataset(X_te_s, y_te), batch_size=256
)

print('Standardized train mean:', X_tr_s.mean(dim=0))
print('Standardized train std: ', X_tr_s.std(dim=0))


## Custom collate (variable length)

A default collate step expects tensors in a batch to have the same shape. Variable-length sequences break that assumption, so a custom collate function pads each batch to a shared length before stacking it.


In [ ]:
class ToySeq(Dataset):
    def __init__(self, rng, n=20):
        self.x = [torch.tensor(rng.integers(1, 10, size=rng.integers(3, 8))) for _ in range(n)]
        self.y = [int(xi.sum() % 2) for xi in self.x]

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        return self.x[i].float(), self.y[i]


def pad_collate(batch):
    xs, ys = zip(*batch)
    max_len = max(x.size(0) for x in xs)
    Xp = torch.zeros(len(xs), max_len, dtype=torch.float32)

    for i, x in enumerate(xs):
        Xp[i, : x.size(0)] = x

    return Xp, torch.tensor(ys, dtype=torch.long)


rng = np.random.default_rng(0)
seq_loader = DataLoader(ToySeq(rng), batch_size=4, collate_fn=pad_collate)
xb, yb = next(iter(seq_loader))
print('Padded batch shape:', xb.shape)
print('Label shape:', yb.shape)


## Visualizations

Each plot answers a slightly different practical question:

- **Train/test scatter:** what data the model is learning from and what data it is evaluated on
- **Decision boundary:** how the trained classifier divides the feature space
- **Standardization plots:** how scaling changes feature values without changing the label structure
- **Padded sequence view:** what a variable-length batch looks like after custom collate padding


In [ ]:
# Plot 1: train and test points in the two-moons dataset
plt.figure(figsize=(5, 4))
plt.scatter(X_tr[:, 0], X_tr[:, 1], c=y_tr, cmap='coolwarm', s=12, alpha=0.8, label='train')
plt.scatter(X_te[:, 0], X_te[:, 1], c=y_te, cmap='coolwarm', s=20, alpha=0.8, marker='x', label='test')
plt.title('Two Moons: Train/Test Split')
plt.xlabel('x1')
plt.ylabel('x2')
plt.legend(loc='best', frameon=False)
plt.tight_layout()
plt.show()


In [ ]:
# Plot 2: decision regions learned by the trained TinyMLP
model.eval()
with torch.no_grad():
    x_min, x_max = float(X_tr[:, 0].min()) - 0.5, float(X_tr[:, 0].max()) + 0.5
    y_min, y_max = float(X_tr[:, 1].min()) - 0.5, float(X_tr[:, 1].max()) + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
    grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
    zz = model(grid).argmax(dim=1).view(xx.shape).cpu().numpy()

plt.figure(figsize=(5, 4))
plt.contourf(xx, yy, zz, levels=1, cmap='coolwarm', alpha=0.3)
plt.scatter(X_tr[:, 0], X_tr[:, 1], c=y_tr, cmap='coolwarm', s=10, alpha=0.9, edgecolors='none')
plt.title('Decision Boundary (TinyMLP)')
plt.xlabel('x1')
plt.ylabel('x2')
plt.tight_layout()
plt.show()


In [ ]:
# Plot 3: feature distributions before and after standardization
Xtr_np = X_tr.numpy()
XtrS_np = X_tr_s.numpy()

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
for i, ax in enumerate(axes):
    ax.hist(Xtr_np[:, i], bins=30, density=True, alpha=0.6, label='raw')
    ax.hist(XtrS_np[:, i], bins=30, density=True, alpha=0.6, label='standardized')
    ax.set_title(f'Feature {i} distribution')
    ax.set_xlabel(f'x{i + 1}')
    ax.set_ylabel('density')
    ax.legend(frameon=False)
fig.suptitle('Standardization Effect on Features', y=1.05)
plt.tight_layout()
plt.show()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3))
ax1.scatter(X_tr[:, 0], X_tr[:, 1], c=y_tr, cmap='coolwarm', s=10, alpha=0.8)
ax1.set_title('Raw features')
ax1.set_xlabel('x1')
ax1.set_ylabel('x2')
ax2.scatter(X_tr_s[:, 0], X_tr_s[:, 1], c=y_tr, cmap='coolwarm', s=10, alpha=0.8)
ax2.set_title('Standardized features')
ax2.set_xlabel('x1 (standardized)')
ax2.set_ylabel('x2 (standardized)')
plt.tight_layout()
plt.show()


In [ ]:
# Plot 4: padded sequence values and true sequence lengths in one batch
lengths = (xb > 0).sum(dim=1).cpu().numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3), gridspec_kw={'width_ratios': [2, 1]})
im = ax1.imshow(xb.cpu().numpy(), aspect='auto', cmap='viridis')
ax1.set_title('Padded batch (values)')
ax1.set_xlabel('time step')
ax1.set_ylabel('sequence index')
plt.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)

ax2.bar(np.arange(len(lengths)), lengths, color='tab:blue')
ax2.set_title('Sequence lengths')
ax2.set_xlabel('batch index')
ax2.set_ylabel('length')
ax2.set_ylim(0, xb.size(1))
plt.tight_layout()
plt.show()


## Chapter 9 Summary

- A `Dataset` stores examples, and a `DataLoader` turns them into mini-batches.
- Training loops become easier to follow when you think in terms of batches, epochs, and repeated optimizer steps.
- A transform such as standardization is a clean way to reuse preprocessing logic.
- Variable-length data needs a custom collate step because the batch must be padded to a common shape.
- Visualizations help check what the data looks like, what the model has learned, and what preprocessing changed.
